In [1]:
# Importações e ínico da sessão spark

import os
from pyspark.sql import SparkSession
from etl_utils import standardize_dates, standardize_ratings, enrich_with_tmdb

spark = SparkSession.builder.appName("MovieETL_Transform").getOrCreate()

In [2]:
# Carregando dados da camada bronze

base_path = "../input"
bronze_path = os.path.join(base_path, "bronze")
silver_path = os.path.join(base_path, "silver")

os.makedirs(silver_path, exist_ok=True)

basics_df = spark.read.parquet(os.path.join(bronze_path, "basics.parquet"))
ratings_df = spark.read.parquet(os.path.join(bronze_path, "ratings.parquet"))

In [3]:
# Join no Datasets

joined_df = basics_df.join(ratings_df, on="tconst", how="left") \
                     .select("tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotes")

In [4]:
# Standarize e Clean nos dados

cleaned_df = standardize_dates(joined_df)
cleaned_df = standardize_ratings(cleaned_df)

In [ ]:
# Enriquecendo com a API do TMDB

cleaned_df.cache()
enriched_df = enrich_with_tmdb(spark, cleaned_df)

enriched_df.write.mode("overwrite").parquet(os.path.join(silver_path, "enriched.parquet"))
print(f"Transformed and enriched {enriched_df.count()} movies to silver.")

In [ ]:
# Parando o Spark

spark.stop()